In [ ]:
from Models.fast_text.fast_text import FastText
from Datasets.datasets import MyDataset
from sklearn.model_selection import KFold
import copy

#### Loading dataset

In [ ]:
import os
os.environ['NationalityDatasets'] = "./data/"

dt = MyDataset()

In [ ]:
data = dt.get_data()
train = dt.get_train()
test = dt.get_test()
validation = dt.get_validation()
train_validation = dt.get_train_validation()

In [ ]:
data.shape, train.shape, test.shape, validation.shape

In [ ]:
data.sample(n=10)

In [ ]:
data['level_2'].value_counts().plot.bar()

In [ ]:
data['level_2'].value_counts().shape

#### Level 2 Classification

In [ ]:
region_2_ft = FastText(data=data, column='level_2')

In [ ]:
region_2_ft.save_train_test(train=train, test=test, validation=validation)

In [ ]:
# %%timeit
# region_2_ft.training()

In [ ]:
# region_2_ft.save_model('level_2_classification.bin')

In [ ]:
region_2_ft.load_model('level_2_classification.bin')

In [ ]:
accuracy = region_2_ft.testing()
print('test accuracy: ', str(accuracy))

In [ ]:
region_2_ft.get_parameters()

In [ ]:
test_df = region_2_ft.load_df()
test_df['pred'] = region_2_ft.predict(test_df['text'])

In [ ]:
region_2_ft.classification_metrics(test_df['label'], test_df['pred'])

Micro F1-score 0.7895622765832234


Macro F1-score 0.6913222591109712


Weighted F1-score 0.7848727550569266


Weighted Precision 0.790018998721864


Weighted Recall 0.7895622765832234


Accuracy Score 0.7895622765832234


Balanced Accuracy Score 0.654463154163547

In [ ]:
region_2_ft.classification_report(test_df['label'], test_df['pred'])

In [ ]:
cm1 = region_2_ft.confusion_matrix(test_df['label'], test_df['pred'], type='precision')
region_2_ft.display_confusion_matrix(cm1)

#### Analysis

In [ ]:
test['label'] = test_df['label'].values
test['pred'] = test_df['pred'].values

In [ ]:
test_df[(test['label'] == 22)].pred.value_counts()

In [ ]:
test_df[(test['label'] == 5)].pred.value_counts()

In [ ]:
test[(test['label'] == 5) & (test['pred'] == 0)].name.sample(n=10)

In [ ]:
for i, country in enumerate(region_2_ft.get_labels()):
    print(i, country)

#### Hierarchical Classification

In [ ]:
fts = []
for i, label in enumerate(region_1_labels):
    print(label)
    
    new_data = data[data.level_1 == label].copy()
    new_train = train[train.level_1 == label].copy()
    new_val = validation[validation.level_1 == label].copy()
    new_test = test[test.level_1 == label].copy()
    
    if len(new_data.level_2.unique()) <= 1:
        new_test['pred'] = new_test.apply(lambda x: ,axis=1)
        continue
    
    new_ft = FastText(data=new_data, type_=label, column='level_2')
    new_ft.save_train_test(train=new_train, test=new_test, validation=new_val)
    
    new_ft.training()
    new_ft.save_model('level_2_' + label+'_classification.bin')
    
    # new_ft.load_model('level_2_' + label + '_classification.bin')
    fts.append(new_ft)
    
    accuracy = new_ft.testing()
    print('test accuracy: ', str(accuracy))
    
    test_df = new_ft.load_df()
    test_df['pred'] = new_ft.predict(test_df['text'])
    
    
    new_ft.classification_metrics(test_df['label'], test_df['pred'])
    
    # new_ft.classification_report(test_df['label'], test_df['pred'])
    
    cm = new_ft.confusion_matrix(test_df['label'], test_df['pred'])
    new_ft.display_confusion_matrix(cm)

In [ ]:
region_1_ft = FastText(column='level_1')
region_1_ft.save_train_test(train=train, test=test, validation=validation)
region_1_labels = region_1_ft.get_labels()
region_1_ft.load_model('level_1_classification.bin')

In [ ]:
data['predicted_level_1'] = region_1_ft.predict_plus_decode(data['name'])
train['predicted_level_1'] = region_1_ft.predict_plus_decode(train['name'])
test['predicted_level_1'] = region_1_ft.predict_plus_decode(test['name'])
validation['predicted_level_1'] = region_1_ft.predict_plus_decode(validation['name'])

In [ ]:
j = 0
new_tests = []
for i, label in enumerate(region_1_labels):
    print(label)
    new_test = test[test.predicted_level_1 == label].copy()
    
    if label in ['English', 'SouthAsia', 'Greek', 'Jewish']:
        new_test['predicted_level_2'] = new_test.apply(lambda x: label if label!='SouthAsia' else 'Indian',axis=1)
        new_tests.append(new_test)
        continue
    
    # new_ft = FastText(type_=label, column='level_2')
    # new_ft = new_ft.load_model('level_2_' + label + '_classification.bin')
    new_test['predicted_level_2'] = fts[j].predict_plus_decode(new_test['name'])
    new_tests.append(new_test)
    j += 1

In [ ]:
import pandas as pd
t = pd.concat(new_tests)

In [ ]:
region_2_ft.classification_metrics(t['level_2'], t['predicted_level_2'])

In [ ]:
region_2_ft.classification_report(t['level_2'], t['predicted_level_2'])

In [ ]:
cm = region_2_ft.confusion_matrix(t['level_2'], t['predicted_level_2'])
region_2_ft.display_confusion_matrix(cm)

In [ ]:
l = set(t['level_2'].value_counts().index)

In [ ]:
s = set(t['predicted_level_2'].value_counts().index)

In [ ]:
s.difference(l)

In [ ]:
test.level_2.unique()

5-fold cross validation

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)  
accuracies = []
accuracies2 = []
parameters = []
f1_micros = []
f1_macros = []
f1_weighteds = []
recalls = []
precisions = []
balanced_accuracies = []
cms = []
models = []
for train_index, val_index in kfold.split(train_validation):
    
    kfold_train = train_validation.iloc[train_index]
    kfold_validation = train_validation.iloc[val_index]
    print(kfold_train.shape, kfold_validation.shape)
    
    region_2_ft = FastText(data=data, column='level_2')
    region_2_ft.save_train_test(train=kfold_train, test=test, validation=kfold_validation)
    
    region_2_ft.training()
    accuracy = region_2_ft.testing()
    print('test accuracy: ', str(accuracy))
    accuracies.append(accuracy)
    parameters.append(region_2_ft.get_parameters())
    
    
    test_df = region_2_ft.load_df()
    test_df['pred'] = region_2_ft.predict(test_df['text'])
    
    region_2_ft.print_classification_metrics(test_df['label'], test_df['pred'])
    
    accuracies2.append(region_2_ft.accuracy_score(test_df['label'], test_df['pred']))
    f1_micros.append(region_2_ft.f1_micro_score(test_df['label'], test_df['pred']))
    f1_macros.append(region_2_ft.f1_macro_score(test_df['label'], test_df['pred']))
    f1_weighteds.append(region_2_ft.f1_weighted_score(test_df['label'], test_df['pred']))
    recalls.append(region_2_ft.recall_score(test_df['label'], test_df['pred']))
    precisions.append(region_2_ft.precision_score(test_df['label'], test_df['pred']))
    balanced_accuracies.append(region_2_ft.balanced_accuracy_score(test_df['label'], test_df['pred']))
    cms.append(region_2_ft.confusion_matrix(test_df['label'], test_df['pred'], type='precision'))
    
    region_2_ft.save_model(f'level_2_classification_{len(accuracies)}.bin')